# Gold: Customer dimension
**Sources:** `silver.crm_customers`, `silver.erp_customers`, `silver.erp_customer_location`  →  **Target:** `gold.dim_customers`

**What this notebook does:**
- Join CRM + ERP customer data using the customer number
- If CRM has no gender, use the ERP gender
- Add a **surrogate key** `customer_key` (1, 2, 3 ...)

In [0]:
CATALOG = "workspace"

## The business logic (SQL)
LEFT JOINs keep every CRM customer, even if ERP has no match.

In [0]:
query = f"""
SELECT
    ROW_NUMBER() OVER (ORDER BY ci.customer_id) AS customer_key,   -- surrogate key
    ci.customer_id,
    ci.customer_number,
    ci.first_name,
    ci.last_name,
    la.country,
    ci.marital_status,
    CASE
        WHEN ci.gender <> 'n/a' THEN ci.gender          -- CRM is the main source
        ELSE COALESCE(ca.gender, 'n/a')                 -- otherwise try ERP
    END AS gender,
    ca.birth_date  AS birthdate,
    ci.created_date AS create_date
FROM {CATALOG}.silver.crm_customers ci
LEFT JOIN {CATALOG}.silver.erp_customers ca
       ON ci.customer_number = ca.customer_number
LEFT JOIN {CATALOG}.silver.erp_customer_location la
       ON ci.customer_number = la.customer_number
"""
df = spark.sql(query)

## Preview

In [0]:
df.display()

## Write the Gold table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.gold.dim_customers")

## Check it Quickly

In [0]:
result = spark.table(f"{CATALOG}.gold.dim_customers")
print("rows:", result.count())